DATA PREPROCESSING

a) Import

In [46]:
import pandas as pd

df = pd.read_csv("Motor_Vehicle_Collisions_-_Crashes_20260401.csv", low_memory=False)
df.shape

(2252143, 29)

b) Parse

In [47]:
# Parsing: Date & Time
df['CRASH DATE'] = pd.to_datetime(df['CRASH DATE'], errors='coerce')
df['CRASH TIME'] = pd.to_datetime(df['CRASH TIME'], format='%H:%M', errors='coerce')

df[['CRASH DATE', 'CRASH TIME']].head()

,CRASH DATE,CRASH TIME
0,2021-09-11,1900-01-01 02:39:00
1,2022-03-26,1900-01-01 11:45:00
2,2023-11-01,1900-01-01 01:29:00
3,2022-06-29,1900-01-01 06:55:00
4,2022-09-21,1900-01-01 13:21:00


In [48]:
# Parsing: Number columns
num_cols = [
    'NUMBER OF PERSONS INJURED',
    'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED',
    'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED',
    'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED',
    'NUMBER OF MOTORIST KILLED',
    'LATITUDE',
    'LONGITUDE'
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df[num_cols].head()

,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,LATITUDE,LONGITUDE
0,2.0,0.0,0,0,0,0,2,0,NaN,NaN
1,1.0,0.0,0,0,0,0,1,0,NaN,NaN
2,1.0,0.0,0,0,0,0,1,0,40.62179,-73.970024
3,0.0,0.0,0,0,0,0,0,0,NaN,NaN
4,0.0,0.0,0,0,0,0,0,0,NaN,NaN


In [49]:
df.dtypes[['CRASH DATE', 'CRASH TIME'] + num_cols]

CRASH DATE                       datetime64[ns]
CRASH TIME                       datetime64[ns]
NUMBER OF PERSONS INJURED               float64
NUMBER OF PERSONS KILLED                float64
NUMBER OF PEDESTRIANS INJURED             int64
NUMBER OF PEDESTRIANS KILLED              int64
NUMBER OF CYCLIST INJURED                 int64
NUMBER OF CYCLIST KILLED                  int64
NUMBER OF MOTORIST INJURED                int64
NUMBER OF MOTORIST KILLED                 int64
LATITUDE                                float64
LONGITUDE                               float64
dtype: object

c) Organize

In [50]:
# Organizing
cols_keep = [
    'CRASH DATE',
    'CRASH TIME',
    'BOROUGH',
    'ZIP CODE',
    'LATITUDE',
    'LONGITUDE',
    'NUMBER OF PERSONS INJURED',
    'NUMBER OF PERSONS KILLED',
    'CONTRIBUTING FACTOR VEHICLE 1',
    'VEHICLE TYPE CODE 1'
]

crash_df = df[cols_keep].copy()
crash_df = crash_df.dropna(subset=['BOROUGH'])
crash_df['HOUR'] = crash_df['CRASH TIME'].dt.hour
crash_df['MONTH'] = crash_df['CRASH DATE'].dt.month

crash_df.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,CONTRIBUTING FACTOR VEHICLE 1,VEHICLE TYPE CODE 1,HOUR,MONTH
2,2023-11-01,1900-01-01 01:29:00,BROOKLYN,11230,40.621790,-73.970024,1.0,0.0,Unspecified,Moped,1,11
9,2021-09-11,1900-01-01 09:35:00,BROOKLYN,11208,40.667202,-73.866500,0.0,0.0,Unspecified,Sedan,9,9
10,2021-12-14,1900-01-01 08:13:00,BROOKLYN,11233,40.683304,-73.917274,0.0,0.0,NaN,NaN,8,12
13,2021-12-14,1900-01-01 08:17:00,BRONX,10475,40.868160,-73.831480,2.0,0.0,Unspecified,Sedan,8,12
14,2021-12-14,1900-01-01 21:10:00,BROOKLYN,11207,40.671720,-73.897100,0.0,0.0,Driver Inexperience,Sedan,21,12


In [51]:
crash_df.shape

(1564912, 12)

BASIC DATA EXPLORATION AND SUMMARY STATISTICS 

1) Descriptive Statistics: The dataset contains over 2.25 million crash records and 29 features, making it a large, high dimensional dataset. Most crashes result in zero or few injuries, indicating a highly skewed distribution. Additionally, crashes are not evenly distributed geographically, with Brooklyn and Queens accounting for the majority of incidents, possibly suggesting population density and traffic volume significantly influence crash frequency.

2. Outlier Detection: The injury distribution is heavily right skewed with extreme outliers, including crashes involving more than 40 injured individuals. These outliers likely represent rare, high impact events and can disproportionately affect statistical measures such as the mean. Therefore, robust statistics or transformations may be necessary for accurate modeling.

3. Hypothesis Testing: With a two sample t test comparing nighttime and daytime crashes yields a statistically significant result (p < 0.001), indicating that nighttime crashes result in a higher number of injuries on average. This suggests that reduced visibility and nighttime driving conditions contribute to increased crash severity.